# Búsqueda de IDs ESIOS correctos

**Objetivo:** Antes de descargar, identificamos exactamente qué IDs corresponden a:
- Generación medida real (NO programada, NO previsión)
- Por tecnología: hidráulica, eólica, solar fotovoltaica, solar térmica, nuclear, ciclo combinado, carbón
- Saldos intercambios Francia, Portugal, Marruecos
- Precios mercado diario
- Frecuencia de red 50 Hz

**Por qué este paso:** Los IDs ESIOS son cientos. Hay 7 versiones de "generación eólica" (programada PBF, programada PHF, prevista, medida, etc.). Solo una es la real medida horaria que necesitamos.

In [1]:
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

TOKEN = "e3c302fe80928e3fbf43b229cf573ccf05b9888e65acd3a1394265f73b1e44be"
HEADERS = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "Host": "api.esios.ree.es",
    "x-api-key": TOKEN
}

BASE_URL = "https://api.esios.ree.es"

print("[OK] Configuración cargada")

[OK] Configuración cargada


## 1. Obtener catálogo completo de indicadores ESIOS

Esta llamada baja todos los ~1.900 indicadores con su descripción.

In [2]:
url = f"{BASE_URL}/indicators"
r = requests.get(url, headers=HEADERS, timeout=120)
r.raise_for_status()
data = r.json()

indicadores = pd.DataFrame(data['indicators'])
print(f"[OK] Total indicadores: {len(indicadores)}")
print(f"[OK] Columnas disponibles: {list(indicadores.columns)}")
indicadores.head(3)

[OK] Total indicadores: 2019
[OK] Columnas disponibles: ['name', 'description', 'short_name', 'id']


,name,description,short_name,id
0,Generación programada PBF Hidráulica UGH,"<p>Es el programa de energía diario, con desgl...",Hidráulica UGH,1
1,Generación programada PBF Hidráulica no UGH,"<p>Es el programa de energía diario, con desgl...",Hidráulica no UGH,2
2,Generación programada PBF Turbinación bombeo,"<p>Es el programa de energía diario, con desgl...",Turbinación bombeo,3


## 2. Búsqueda por palabras clave

In [3]:
def buscar(palabras_clave, en='name', exclude=None):
    """Busca indicadores por palabra clave en columna 'name' o 'short_name'."""
    df = indicadores.copy()
    for kw in palabras_clave:
        df = df[df[en].str.contains(kw, case=False, na=False, regex=False)]
    if exclude:
        for ex in exclude:
            df = df[~df[en].str.contains(ex, case=False, na=False, regex=False)]
    return df[['id', 'name', 'short_name']].reset_index(drop=True)

# Buscar generación MEDIDA (no programada, no prevista)
print("=== GENERACIÓN MEDIDA HIDRÁULICA ===")
print(buscar(['generación', 'medida', 'hidráulica'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())

=== GENERACIÓN MEDIDA HIDRÁULICA ===
      id                                 name         short_name
0   1150     Generación medida Hidráulica UGH     Hidráulica UGH
1   1151  Generación medida Hidráulica no UGH  Hidráulica no UGH
2  10035         Generación medida Hidráulica         Hidráulica


In [4]:
print("=== GENERACIÓN MEDIDA EÓLICA ===")
print(buscar(['generación', 'medida', 'eólica'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())
print()
print("=== GENERACIÓN MEDIDA SOLAR FOTOVOLTAICA ===")
print(buscar(['generación', 'medida', 'fotovoltaica'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())
print()
print("=== GENERACIÓN MEDIDA SOLAR TÉRMICA ===")
print(buscar(['generación', 'medida', 'solar térmica'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())

=== GENERACIÓN MEDIDA EÓLICA ===
      id                                name        short_name
0   1159  Generación medida Eólica terrestre  Eólica terrestre
1   1160     Generación medida Eólica marina     Eólica marina
2  10037            Generación medida Eólica            Eólica

=== GENERACIÓN MEDIDA SOLAR FOTOVOLTAICA ===
     id                                  name          short_name
0  1161  Generación medida Solar fotovoltaica  Solar fotovoltaica

=== GENERACIÓN MEDIDA SOLAR TÉRMICA ===
     id                             name     short_name
0  1162  Generación medida Solar térmica  Solar térmica


In [5]:
print("=== GENERACIÓN MEDIDA NUCLEAR ===")
print(buscar(['generación', 'medida', 'nuclear'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())
print()
print("=== GENERACIÓN MEDIDA CICLO COMBINADO ===")
print(buscar(['generación', 'medida', 'ciclo combinado'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())
print()
print("=== GENERACIÓN MEDIDA CARBÓN ===")
print(buscar(['generación', 'medida', 'carbón'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())
print(buscar(['generación', 'medida', 'hulla'], exclude=['programada', 'prevista', 'PBF', 'PHF']).to_string())

=== GENERACIÓN MEDIDA NUCLEAR ===
     id                       name short_name
0  1153  Generación medida Nuclear    Nuclear

=== GENERACIÓN MEDIDA CICLO COMBINADO ===
     id                               name       short_name
0  1156  Generación medida Ciclo combinado  Ciclo combinado

=== GENERACIÓN MEDIDA CARBÓN ===
      id                                               name                       short_name
0   1165  Generación medida Derivados del petróleo ó carbón  Derivados del petróleo ó carbón
1  10036                           Generación medida Carbón                           Carbón
     id                                    name            short_name
0  1154       Generación medida Hulla antracita       Hulla antracita
1  1155  Generación medida Hulla sub-bituminosa  Hulla sub-bituminosa


In [6]:
# INTERCAMBIOS INTERNACIONALES
print("=== INTERCAMBIOS PORTUGAL ===")
print(buscar(['portugal', 'saldo']).to_string())
print()
print("=== INTERCAMBIOS FRANCIA ===")
print(buscar(['francia', 'saldo']).to_string())
print()
print("=== INTERCAMBIOS MARRUECOS ===")
print(buscar(['marruecos', 'saldo']).to_string())

=== INTERCAMBIOS PORTUGAL ===
       id                                                                 name                       short_name
0     557  Saldo horario de interconexión con Portugal importación telemedidas             Portugal importación
1     561  Saldo horario de interconexión con Portugal exportación telemedidas             Portugal exportación
2   10014                             Generación programada P48 Saldo Portugal                   Saldo Portugal
3   10044                                     Generación medida Saldo Portugal                   Saldo Portugal
4   10113                             Generación programada PBF Saldo Portugal                   Saldo Portugal
5   10114                             Generación programada PVP Saldo Portugal                   Saldo Portugal
6   10115                            Generación programada PHF1 Saldo Portugal                   Saldo Portugal
7   10116                            Generación programada PHF2 Saldo Port

In [7]:
# PRECIOS MERCADO DIARIO
print("=== PRECIO MERCADO DIARIO ===")
print(buscar(['precio', 'mercado', 'diario']).to_string())
print()
print("=== PVPC ===")
print(buscar(['PVPC']).to_string())

=== PRECIO MERCADO DIARIO ===
      id                                                                                                                                               name                                                                                                                                         short_name
0    600                                                                                                                         Precio mercado SPOT Diario                                                                                                                                       Mercado SPOT
1    612                                                                                                           Precio mercado SPOT Intradiario Sesión 1                                                                                                                               Intradiario Sesión 1
2    613                                                     

In [8]:
# FRECUENCIA DE RED 50 Hz
print("=== FRECUENCIA DE RED ===")
print(buscar(['frecuencia']).to_string())
print()
print("=== INERCIA / REGULACIÓN ===")
print(buscar(['regulación', 'primaria']).to_string())
print(buscar(['regulación', 'secundaria']).to_string())
print(buscar(['regulación', 'terciaria']).to_string())

=== FRECUENCIA DE RED ===
Empty DataFrame
Columns: [id, name, short_name]
Index: []

=== INERCIA / REGULACIÓN ===
Empty DataFrame
Columns: [id, name, short_name]
Index: []
       id                                                                         name                                short_name
0     630                      Requerimientos reserva de regulación secundaria a subir               Regulación secundaria subir
1     631                      Requerimientos reserva de regulación secundaria a bajar               Regulación secundaria bajar
2     632                          Asignación reserva de regulación secundaria a subir               Regulación secundaria subir
3     633                          Asignación reserva de regulación secundaria a bajar               Regulación secundaria bajar
4     634                              Precio reserva de regulación secundaria a bajar                Reserva secundaria a subir
5     680                Energía activada  de regulaci

In [9]:
# DEMANDA REAL HORARIA
print("=== DEMANDA REAL ===")
print(buscar(['demanda', 'real']).to_string())
print()
print("=== DEMANDA PROGRAMADA ===")
print(buscar(['demanda', 'programada']).to_string())
print()
print("=== DEMANDA PREVISTA ===")
print(buscar(['demanda', 'prevista']).to_string())

=== DEMANDA REAL ===
      id                                                                                                                   name                        short_name
0    624                                                                                             Demanda real máximo diario                     Máximo diario
1    625                                                                                             Demanda real mínimo diario                     Mínimo diario
2    725  Coste unitario soportado por la demanda del proceso de solución de Restricciones Técnicas de Seguridad en Tiempo Real                  Restricciones TR
3   1293                                                                                                           Demanda real                      Demanda real
4   1740                                                                                                       Demanda Real SNP                  Demanda Real SNP
5   203

In [10]:
# GUARDAR CATÁLOGO COMPLETO PARA CONSULTAS POSTERIORES
from pathlib import Path

PROYECTO_DIR = Path(r"C:\Users\Hector\Desktop\premios-steam-2026")
ESIOS_DIR = PROYECTO_DIR / "datos" / "esios_raw"
ESIOS_DIR.mkdir(parents=True, exist_ok=True)

indicadores.to_csv(ESIOS_DIR / "_catalogo_completo_esios.csv", index=False)
print(f"[OK] Catálogo completo guardado en: {ESIOS_DIR / '_catalogo_completo_esios.csv'}")
print(f"     Total: {len(indicadores)} indicadores")

[OK] Catálogo completo guardado en: C:\Users\Hector\Desktop\premios-steam-2026\datos\esios_raw\_catalogo_completo_esios.csv
     Total: 2019 indicadores


## 3. ACCIÓN — Compártelo conmigo

Una vez ejecutado este notebook, **mándame el output completo de las búsquedas**.

Yo identificaré los IDs exactos correctos y generaré el notebook definitivo de descarga con IDs validados.

**Lo que necesito ver:**
1. Lista de IDs de generación medida (cada tecnología).
2. ID del saldo Portugal correcto.
3. ID de frecuencia de red 50 Hz (NUEVO — no estaba en la primera descarga).
4. ID de reservas regulación primaria/secundaria/terciaria (NUEVO).
5. ID precios mercado diario.